In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append('../')
print(sys.path[-3:])

['', '/home/hieutt/miniconda3/envs/torchtf/lib/python3.9/site-packages', '../']


In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report


def evaluate_from_cm(cm, target_names, model_name="Model", digits=4, print_result=True):
    """
    Evaluate classification performance directly from a confusion matrix.

    Args:
        cm: np.ndarray, shape [num_classes, num_classes]
            Confusion matrix where rows are true labels and columns are predicted labels.
        target_names: list[str]
            Class names in the same order as the confusion matrix.
        model_name: str
            Model name used for printing.
        digits: int
            Number of decimal digits in sklearn classification_report.
        print_result: bool
            Whether to print classification report and average FNR.

    Returns:
        result: dict
            Contains report_text, report_dict, fnr_dict, fnr_df, accuracy,
            macro_fnr, macro_f1, weighted_f1, etc.
    """

    cm = np.asarray(cm)
    num_classes = cm.shape[0]

    if cm.shape[0] != cm.shape[1]:
        raise ValueError("Confusion matrix must be square.")

    if len(target_names) != num_classes:
        raise ValueError("Length of target_names must match number of classes in cm.")

    # Instead of expanding y_true/y_pred, use sample_weight.
    true_idx, pred_idx = np.indices(cm.shape)
    y_true = true_idx.ravel()
    y_pred = pred_idx.ravel()
    weights = cm.ravel()

    report_text = classification_report(
        y_true,
        y_pred,
        sample_weight=weights,
        target_names=target_names,
        digits=digits,
        zero_division=0
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        sample_weight=weights,
        target_names=target_names,
        digits=digits,
        output_dict=True,
        zero_division=0
    )

    # False Negative Rate per class
    fnr_dict = {}
    fnr_rows = []

    for i, name in enumerate(target_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP

        if TP + FN > 0:
            fnr = FN / (TP + FN)
        else:
            fnr = 0.0

        fnr_dict[name] = fnr

        fnr_rows.append({
            "class": name,
            "TP": int(TP),
            "FN": int(FN),
            "FNR": fnr,
            "FNR (%)": fnr * 100
        })

    fnr_df = pd.DataFrame(fnr_rows)
    
    # Calculate Macro Average FNR
    macro_fnr = fnr_df["FNR"].mean()

    result = {
        "model_name": model_name,
        "report_text": report_text,
        "report_dict": report_dict,
        "fnr_dict": fnr_dict,
        "fnr_df": fnr_df,
        "macro_fnr": macro_fnr,
        "accuracy": report_dict["accuracy"],
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
        "macro_precision": report_dict["macro avg"]["precision"],
        "macro_recall": report_dict["macro avg"]["recall"],
        "weighted_precision": report_dict["weighted avg"]["precision"],
        "weighted_recall": report_dict["weighted avg"]["recall"],
    }

    if print_result:
        print(f"\nClassification Report: {model_name}")
        print(report_text)

        # Print only the average FNR
        print(f"Average False Negative Rate (Macro FNR): {macro_fnr * 100:.4f}%\n")

    return result

In [3]:
target_names = [
    "Normal",
    "Combined",
    "DoS",
    "Fuzzy",
    "Gear",
    "Interval",
    "RPM",
    "Speed",
    "Standstill",
    "Systematic"
]

## Confusion matrix

In [4]:
baseline_cms = {}

In [ ]:
baseline_cms["GCN"] = np.array([
    [1944989,    1614,      34,      51,     270,     200,     152,     435,      48,      10],
    [   2067,   14750,       0,       0,       0,      17,      70,      66,       0,       0],
    [      4,       0,   10228,       0,       0,       0,       0,       0,       0,       0],
    [    220,       0,       0,    6943,       0,       0,       0,       0,       0,     218],
    [     97,       0,       0,       0,     815,       0,       0,       0,       2,       0],
    [     17,       0,       0,       0,       0,   21062,       0,      23,       0,       0],
    [   1051,       0,       0,       0,       0,       0,    2305,       0,       0,       0],
    [    220,      16,       0,       0,       0,      17,       0,    3396,       0,       0],
    [     20,       0,       0,       0,       0,       0,       0,       0,    2161,       0],
    [    207,       0,       0,     882,       0,       0,       0,       0,       0,    3755]
])

baseline_cms["R-GCN"] = np.array([
    [1945669,    1365,      27,      62,     150,     106,     168,     179,      47,      30],
    [   1215,   15749,       0,       0,       0,       3,       2,       1,       0,       0],
    [      1,       0,   10229,       1,       0,       1,       0,       0,       0,       0],
    [    174,       0,       0,    7073,       0,       0,       0,       0,       0,     134],
    [    228,       0,       0,       0,     686,       0,       0,       0,       0,       0],
    [     28,       0,       0,       0,       0,   21074,       0,       0,       0,       0],
    [    483,      12,       0,       0,       0,       0,    2861,       0,       0,       0],
    [    249,      45,       0,       0,       0,       6,       0,    3349,       0,       0],
    [     29,       0,       0,       0,       0,       0,       0,       0,    2152,       0],
    [    167,       0,       0,     260,       0,       1,       0,       0,       0,    4416]
])

baseline_cms["GIN"] = np.array([
    [1945408,     946,     185,      64,     375,     182,     295,     186,     110,      52],
    [   1855,   15108,       0,       0,       0,       0,       5,       2,       0,       0],
    [      2,       0,   10230,       0,       0,       0,       0,       0,       0,       0],
    [    140,       0,       0,    7109,       0,       0,       0,       0,       0,     132],
    [     50,       0,       0,       0,     864,       0,       0,       0,       0,       0],
    [     33,       4,       0,       0,       0,   21064,       0,       1,       0,       0],
    [    631,       2,       0,       0,       0,       0,    2723,       0,       0,       0],
    [    478,      82,       0,       0,       0,       4,       0,    3085,       0,       0],
    [     12,       0,       0,       0,       0,       0,       0,       0,    2169,       0],
    [    134,       0,       0,      40,       0,       1,       0,       0,       0,    4669]
])

baseline_cms["GATv2"] = np.array([
    [1946351,     772,      11,      60,     219,     115,      93,     107,      43,      32],
    [   1520,   15438,       0,       0,       0,       3,       4,       5,       0,       0],
    [      1,       0,   10230,       0,       0,       1,       0,       0,       0,       0],
    [    160,       0,       0,    7098,       0,       0,       0,       0,       0,     123],
    [     28,       0,       0,       0,     886,       0,       0,       0,       0,       0],
    [     15,       0,       0,       0,       0,   21087,       0,       0,       0,       0],
    [    338,       3,       0,       0,       0,       0,    3015,       0,       0,       0],
    [    259,      15,       0,       0,       0,       5,       0,    3370,       0,       0],
    [      5,       0,       0,       0,       0,       0,       0,       0,    2176,       0],
    [    129,       0,       0,     174,       1,       0,       0,       0,       0,    4540]
])

baseline_cms["GraphSAGE"] = np.array([
    [1945754,    1143,       5,      44,     178,     206,     152,     293,       1,      27],
    [   1931,   14976,       0,       0,       0,       6,      27,      30,       0,       0],
    [      2,       0,   10230,       0,       0,       0,       0,       0,       0,       0],
    [    208,       0,       0,    6906,       0,       0,       0,       0,       0,     267],
    [     79,       0,       0,       0,     835,       0,       0,       0,       0,       0],
    [      5,       0,       0,       0,       0,   21093,       0,       4,       0,       0],
    [    649,       2,       0,       0,       0,       0,    2705,       0,       0,       0],
    [    248,       4,       0,       0,       0,      29,       0,    3368,       0,       0],
    [      2,       0,       0,       0,       0,       0,       0,       0,    2179,       0],
    [    164,       0,       0,     182,       0,       0,       0,       0,       0,    4498]
])

baseline_cms["Edge-GATv2"] = np.array([
    [1946119,    1189,       5,      44,     100,      74,      61,     186,       1,      24],
    [    568,   16395,       0,       0,       0,       0,       0,       7,       0,       0],
    [      1,       0,   10231,       0,       0,       0,       0,       0,       0,       0],
    [    107,       0,       0,    7179,       0,       0,       0,       0,       0,      95],
    [      0,       0,       0,       0,     914,       0,       0,       0,       0,       0],
    [      4,       0,       0,       0,       0,   21096,       0,       2,       0,       0],
    [     61,       6,       0,       0,       0,       0,    3289,       0,       0,       0],
    [    102,       7,       0,       0,       0,      10,       0,    3530,       0,       0],
    [      1,       0,       0,       0,       0,       0,       0,       0,    2180,       0],
    [    100,       0,       0,      62,       0,       0,       0,       0,       0,    4682]
])

baseline_cms["Graph-Transformer"] = np.array([
    [1946609,     878,       0,      51,       2,      64,      45,     103,       8,      43],
    [    692,   16277,       0,       0,       0,       0,       0,       1,       0,       0],
    [      1,       0,   10231,       0,       0,       0,       0,       0,       0,       0],
    [    107,       0,       0,    7165,       0,       0,       0,       0,       0,     109],
    [      4,       0,       0,       0,     910,       0,       0,       0,       0,       0],
    [      1,       0,       0,       0,       0,   21098,       0,       3,       0,       0],
    [     95,       1,       0,       0,       0,       0,    3260,       0,       0,       0],
    [    123,      28,       0,       0,       0,       2,       0,    3496,       0,       0],
    [      0,       0,       0,       0,       0,       0,       0,       0,    2181,       0],
    [     92,       0,       0,      13,       0,       0,       0,       0,       0,    4739]
])

baseline_cms["2011_chevrolet_impala_graph"] = np.array([
    [16975,    27,     0,     2,     2,     0,     6,     7,     0,     0],
    [   24,  1851,     0,     0,     0,     0,     0,     0,     0,     0],
    [    1,     0,  1111,     0,     0,     0,     0,     0,     0,     0],
    [    2,     0,     0,  1923,     0,     0,     0,     0,     0,     0],
    [    2,     0,     0,     0,   655,     0,     0,     0,     0,     0],
    [    0,     0,     0,     0,     0,  1089,     0,     0,     0,     0],
    [   12,     0,     0,     0,     0,     0,   935,     0,     0,     0],
    [    1,     0,     0,     0,     0,     0,     0,  1413,     0,     0],
    [    0,     0,     0,     0,     0,     0,     0,     0,   505,     0],
    [    2,     0,     0,     1,     0,     0,     0,     0,     0,   906]
])

baseline_cms["2011_chevrolet_impala_node"] = np.array([
    [1706493,   632,     1,    36,    30,     4,     9,    88,     0,    24],
    [    694,  4723,     0,     0,     1,     0,     0,     1,     1,     0],
    [      1,     0, 18161,     0,     0,     0,     0,     0,     0,     0],
    [     71,     0,     0, 10366,     0,     0,     1,     1,     2,    41],
    [      8,     0,     0,     0,  1880,     0,     0,     0,     0,     0],
    [      0,     0,     0,     0,     0,  1450,     0,     0,     0,     0],
    [     16,     0,     0,     0,     0,     0,  2121,     0,     0,     0],
    [     20,     4,     0,     0,     0,     2,     0,  2798,     0,     0],
    [      0,     0,     0,     0,     0,     0,     0,     0,   984,     0],
    [     37,     0,     0,    52,     1,     0,     1,     0,     0,  6173]
])

baseline_cms["2011_chevrolet_traverse_graph"] = np.array([
    [30526,    13,     0,     1,    19,     9,    10,     5,     0,     0],
    [   10,  3224,     0,     0,     0,     0,     0,     0,     0,     0],
    [    0,     0,  1904,     0,     0,     0,     0,     0,     0,     0],
    [    4,     0,     0,  4717,     0,     0,     0,     0,     0,     0],
    [    5,     0,     0,     0,   238,     0,     0,     0,     0,     0],
    [    0,     0,     0,     0,     0,  3278,     0,     0,     0,     0],
    [   10,     0,     0,     0,     0,     0,   157,     0,     0,     0],
    [   15,     0,     0,     0,     0,     0,     0,   261,     0,     0],
    [    1,     0,     0,     0,     0,     0,     0,     0,   120,     0],
    [    4,     0,     0,     0,     0,     0,     0,     0,     0,  1245]
])

baseline_cms["2011_chevrolet_traverse_node"] = np.array([
    [2864335,   791,     0,    78,    13,   143,    23,    21,     2,    39],
    [    272, 10285,     0,     0,    12,     0,     0,     0,     2,     0],
    [      0,     0, 15895,     0,     0,     0,     0,     0,     0,     0],
    [    177,     2,     0, 24970,     0,     1,     0,     0,     0,    18],
    [     11,     1,     0,     0,   221,     0,     0,     0,     0,     0],
    [      3,     1,     0,     0,     0,  3283,     0,     0,     0,     0],
    [     21,     0,     0,     0,     0,     0,   167,     0,     0,     0],
    [     25,     1,     0,     0,     0,     3,     0,   637,     0,     0],
    [      2,     1,     0,     0,     0,     0,     0,     0,   118,     0],
    [     66,     0,     0,    10,     0,     1,     0,     0,     0,  8003]
])

baseline_cms["2016_chevrolet_silverado_graph"] = np.array([
    [19465,    48,     0,     2,     4,     1,     7,    20,     0,     4],
    [   29,  1015,     0,     0,     0,     0,     0,     0,     0,     0],
    [    0,     0,  5262,     0,     2,     0,     0,     0,     0,     0],
    [    2,     0,     0,  1895,     0,     0,     0,     0,     0,     0],
    [    1,     0,     0,     0,   303,     0,     0,     0,     0,     0],
    [    0,     0,     0,     0,     0,  1166,     0,     0,     0,     0],
    [    7,     0,     0,     0,     0,     0,   246,     0,     0,     0],
    [    3,     0,     0,     0,     0,     0,     0,   999,     0,     0],
    [    0,     0,     0,     0,     0,     0,     0,     0,   274,     0],
    [   26,     0,     0,     0,     0,     0,     0,     0,     1,  2435]
])

baseline_cms["2016_chevrolet_silverado_node"] = np.array([
    [2051030,   173,    11,    61,    57,     5,    14,    80,     8,    33],
    [    169,  2159,     0,     0,     0,     0,     0,     0,     0,     0],
    [      0,     0, 52048,     0,     0,     0,     0,     0,     0,     0],
    [    108,     0,     0, 10822,     1,     0,     0,     0,     0,    90],
    [     10,     0,     0,     0,   758,     0,     0,     0,     0,     0],
    [      0,     0,     0,     0,     0,  1992,     0,     0,     0,     0],
    [      7,     0,     0,     0,     0,     0,   371,     0,     4,     0],
    [     21,     0,     0,     0,     0,     0,     0,  1117,     0,     0],
    [      5,     0,     0,     0,     0,     0,     0,     0,   485,     0],
    [     79,     1,     0,   101,     1,     0,     0,     0,     2,  4045]
])

## GCN


In [ ]:
graph_sage_cm = baseline_cms["GCN"]
result = evaluate_from_cm(
    cm=graph_sage_cm,
    target_names=target_names,
    model_name="GCN",
    digits=4,
    print_result=True
)

## GraphSAGE

In [6]:
graph_sage_cm = baseline_cms["GraphSAGE"]
result = evaluate_from_cm(
    cm=graph_sage_cm,
    target_names=target_names,
    model_name="GraphSAGE",
    digits=4,
    print_result=True
)


Classification Report: GraphSAGE
              precision    recall  f1-score   support

      Normal     0.9983    0.9989    0.9986 1947803.0
    Combined     0.9287    0.8825    0.9050   16970.0
         DoS     0.9995    0.9998    0.9997   10232.0
       Fuzzy     0.9683    0.9356    0.9517    7381.0
        Gear     0.8243    0.9136    0.8666     914.0
    Interval     0.9887    0.9996    0.9941   21102.0
         RPM     0.9379    0.8060    0.8670    3356.0
       Speed     0.9115    0.9230    0.9172    3649.0
  Standstill     0.9995    0.9991    0.9993    2181.0
  Systematic     0.9386    0.9286    0.9336    4844.0

    accuracy                         0.9971 2018432.0
   macro avg     0.9495    0.9387    0.9433 2018432.0
weighted avg     0.9970    0.9971    0.9970 2018432.0

Average False Negative Rate (Macro FNR): 6.1330%



## R-GCN

In [ ]:
graph_sage_cm = baseline_cms["R-GCN"]
result = evaluate_from_cm(
    cm=graph_sage_cm,
    target_names=target_names,
    model_name="R-GCN",
    digits=4,
    print_result=True
)

## GIN

In [ ]:
graph_sage_cm = baseline_cms["GIN"]
result = evaluate_from_cm(
    cm=graph_sage_cm,
    target_names=target_names,
    model_name="GIN",
    digits=4,
    print_result=True
)

## GATv2

In [ ]:
graph_sage_cm = baseline_cms["GATv2"]
result = evaluate_from_cm(
    cm=graph_sage_cm,
    target_names=target_names,
    model_name="GATv2",
    digits=4,
    print_result=True
)

## Graph Edge-GATv2

In [7]:
edge_gat_v2_cm = baseline_cms["Edge-GATv2"]
result = evaluate_from_cm(
    cm=edge_gat_v2_cm,
    target_names=target_names,
    model_name="Edge-GATv2",
    digits=4,
    print_result=True
)


Classification Report: Edge-GATv2
              precision    recall  f1-score   support

      Normal     0.9995    0.9991    0.9993 1947803.0
    Combined     0.9317    0.9661    0.9486   16970.0
         DoS     0.9995    0.9999    0.9997   10232.0
       Fuzzy     0.9854    0.9726    0.9790    7381.0
        Gear     0.9014    1.0000    0.9481     914.0
    Interval     0.9960    0.9997    0.9979   21102.0
         RPM     0.9818    0.9800    0.9809    3356.0
       Speed     0.9477    0.9674    0.9574    3649.0
  Standstill     0.9995    0.9995    0.9995    2181.0
  Systematic     0.9752    0.9666    0.9709    4844.0

    accuracy                         0.9986 2018432.0
   macro avg     0.9718    0.9851    0.9781 2018432.0
weighted avg     0.9986    0.9986    0.9986 2018432.0

Average False Negative Rate (Macro FNR): 1.4898%



## Graph-Transformer

In [6]:
graph_transformer_cm = baseline_cms["Graph-Transformer"]
result = evaluate_from_cm(
    cm=graph_transformer_cm,
    target_names=target_names,
    model_name="Graph-Transformer",
    digits=4,
    print_result=True
)


Classification Report: Graph-Transformer
              precision    recall  f1-score   support

      Normal     0.9994    0.9994    0.9994 1947803.0
    Combined     0.9472    0.9592    0.9532   16970.0
         DoS     1.0000    0.9999    1.0000   10232.0
       Fuzzy     0.9911    0.9707    0.9808    7381.0
        Gear     0.9978    0.9956    0.9967     914.0
    Interval     0.9969    0.9998    0.9983   21102.0
         RPM     0.9864    0.9714    0.9788    3356.0
       Speed     0.9703    0.9581    0.9641    3649.0
  Standstill     0.9963    1.0000    0.9982    2181.0
  Systematic     0.9689    0.9783    0.9736    4844.0

    accuracy                         0.9988 2018432.0
   macro avg     0.9854    0.9832    0.9843 2018432.0
weighted avg     0.9988    0.9988    0.9988 2018432.0

Average False Negative Rate (Macro FNR): 1.6759%



## 2011-chevrolet-impala

In [ ]:
graph_transformer_cm = baseline_cms["2011_chevrolet_impala_graph"]
result = evaluate_from_cm(
    cm=graph_transformer_cm,
    target_names=target_names,
    model_name="2011_chevrolet_impala_graph",
    digits=4,
    print_result=True
)


Classification Report: 2011_chevrolet_impala
              precision    recall  f1-score   support

      Normal     0.9974    0.9974    0.9974   17019.0
    Combined     0.9856    0.9872    0.9864    1875.0
         DoS     1.0000    0.9991    0.9996    1112.0
       Fuzzy     0.9984    0.9990    0.9987    1925.0
        Gear     0.9970    0.9970    0.9970     657.0
    Interval     1.0000    1.0000    1.0000    1089.0
         RPM     0.9936    0.9873    0.9905     947.0
       Speed     0.9951    0.9993    0.9972    1414.0
  Standstill     1.0000    1.0000    1.0000     505.0
  Systematic     1.0000    0.9967    0.9983     909.0

    accuracy                         0.9968   27452.0
   macro avg     0.9967    0.9963    0.9965   27452.0
weighted avg     0.9968    0.9968    0.9968   27452.0

Average False Negative Rate (Macro FNR): 0.3705%



In [10]:
graph_transformer_cm = baseline_cms["2011_chevrolet_impala_node"]
result = evaluate_from_cm(
    cm=graph_transformer_cm,
    target_names=target_names,
    model_name="2011_chevrolet_impala_node",
    digits=4,
    print_result=True
)


Classification Report: 2011_chevrolet_impala_node
              precision    recall  f1-score   support

      Normal     0.9995    0.9995    0.9995 1707317.0
    Combined     0.8813    0.8714    0.8763    5420.0
         DoS     0.9999    0.9999    0.9999   18162.0
       Fuzzy     0.9916    0.9889    0.9903   10482.0
        Gear     0.9833    0.9958    0.9895    1888.0
    Interval     0.9959    1.0000    0.9979    1450.0
         RPM     0.9948    0.9925    0.9937    2137.0
       Speed     0.9688    0.9908    0.9797    2824.0
  Standstill     0.9970    1.0000    0.9985     984.0
  Systematic     0.9896    0.9855    0.9875    6264.0

    accuracy                         0.9990 1756928.0
   macro avg     0.9802    0.9824    0.9813 1756928.0
weighted avg     0.9990    0.9990    0.9990 1756928.0

Average False Negative Rate (Macro FNR): 1.7566%



## 2021-chevrolet-traverse

In [8]:
graph_transformer_cm = baseline_cms["2011_chevrolet_traverse_graph"]
result = evaluate_from_cm(
    cm=graph_transformer_cm,
    target_names=target_names,
    model_name="2011_chevrolet_traverse_graph",
    digits=4,
    print_result=True
)


Classification Report: 2011_chevrolet_traverse_graph
              precision    recall  f1-score   support

      Normal     0.9984    0.9981    0.9983   30583.0
    Combined     0.9960    0.9969    0.9964    3234.0
         DoS     1.0000    1.0000    1.0000    1904.0
       Fuzzy     0.9998    0.9992    0.9995    4721.0
        Gear     0.9261    0.9794    0.9520     243.0
    Interval     0.9973    1.0000    0.9986    3278.0
         RPM     0.9401    0.9401    0.9401     167.0
       Speed     0.9812    0.9457    0.9631     276.0
  Standstill     1.0000    0.9917    0.9959     121.0
  Systematic     1.0000    0.9968    0.9984    1249.0

    accuracy                         0.9977   45776.0
   macro avg     0.9839    0.9848    0.9842   45776.0
weighted avg     0.9977    0.9977    0.9977   45776.0

Average False Negative Rate (Macro FNR): 1.5207%



In [14]:
graph_transformer_cm = baseline_cms["2011_chevrolet_traverse_node"]
result = evaluate_from_cm(
    cm=graph_transformer_cm,
    target_names=target_names,
    model_name="2011_chevrolet_traverse_node",
    digits=4,
    print_result=True
)


Classification Report: 2011_chevrolet_traverse_node
              precision    recall  f1-score   support

      Normal     0.9998    0.9996    0.9997 2865445.0
    Combined     0.9281    0.9729    0.9500   10571.0
         DoS     1.0000    1.0000    1.0000   15895.0
       Fuzzy     0.9965    0.9921    0.9943   25168.0
        Gear     0.8984    0.9485    0.9228     233.0
    Interval     0.9569    0.9988    0.9774    3287.0
         RPM     0.8789    0.8883    0.8836     188.0
       Speed     0.9681    0.9565    0.9622     666.0
  Standstill     0.9672    0.9752    0.9712     121.0
  Systematic     0.9929    0.9905    0.9917    8080.0

    accuracy                         0.9994 2929654.0
   macro avg     0.9587    0.9722    0.9653 2929654.0
weighted avg     0.9994    0.9994    0.9994 2929654.0

Average False Negative Rate (Macro FNR): 2.7760%



## 2016-chevrolet-silverado

In [17]:
graph_transformer_cm = baseline_cms["2016_chevrolet_silverado_graph"]
result = evaluate_from_cm(
    cm=graph_transformer_cm,
    target_names=target_names,
    model_name="2016_chevrolet_silverado_graph",
    digits=4,
    print_result=True
)


Classification Report: 2016_chevrolet_silverado_graph
              precision    recall  f1-score   support

      Normal     0.9965    0.9956    0.9961   19551.0
    Combined     0.9548    0.9722    0.9635    1044.0
         DoS     1.0000    0.9996    0.9998    5264.0
       Fuzzy     0.9989    0.9989    0.9989    1897.0
        Gear     0.9806    0.9967    0.9886     304.0
    Interval     0.9991    1.0000    0.9996    1166.0
         RPM     0.9723    0.9723    0.9723     253.0
       Speed     0.9804    0.9970    0.9886    1002.0
  Standstill     0.9964    1.0000    0.9982     274.0
  Systematic     0.9984    0.9890    0.9937    2462.0

    accuracy                         0.9953   33217.0
   macro avg     0.9877    0.9921    0.9899   33217.0
weighted avg     0.9953    0.9953    0.9953   33217.0

Average False Negative Rate (Macro FNR): 0.7853%



In [23]:
graph_transformer_cm = baseline_cms["2016_chevrolet_silverado_node"]
result = evaluate_from_cm(
    cm=graph_transformer_cm,
    target_names=target_names,
    model_name="2016_chevrolet_silverado_node",
    digits=4,
    print_result=True
)


Classification Report: 2016_chevrolet_silverado_node
              precision    recall  f1-score   support

      Normal     0.9998    0.9998    0.9998 2051472.0
    Combined     0.9254    0.9274    0.9264    2328.0
         DoS     0.9998    1.0000    0.9999   52048.0
       Fuzzy     0.9853    0.9819    0.9836   11021.0
        Gear     0.9278    0.9870    0.9565     768.0
    Interval     0.9975    1.0000    0.9987    1992.0
         RPM     0.9636    0.9712    0.9674     382.0
       Speed     0.9332    0.9815    0.9567    1138.0
  Standstill     0.9719    0.9898    0.9808     490.0
  Systematic     0.9705    0.9565    0.9634    4229.0

    accuracy                         0.9995 2125868.0
   macro avg     0.9675    0.9795    0.9733 2125868.0
weighted avg     0.9995    0.9995    0.9995 2125868.0

Average False Negative Rate (Macro FNR): 2.0485%

